In [ ]:
%pip -q install openai pandas

## 1. Configuration and original prompts


In [ ]:

from pathlib import Path
from getpass import getpass
import hashlib
import json
import os
import re
import time
import zipfile
import openai
from openai import OpenAI
import pandas as pd

# ==================== EXPERIMENT CONFIGURATION ====================

APPROACH = "---" #stratified | Random

# Supported systems: bgl, apache, linux, zookeeper
SYSTEM_KEY = "---"

SUMMARY_TYPE = "---"  # extractive, abstractive, hybrid
USE_EXPLANATION = True

RUN_TAG = (
    SYSTEM_KEY + "_" + SUMMARY_TYPE
    + ("_explanation" if USE_EXPLANATION else "")
)

# ==================== INPUT / OUTPUT ====================

INPUT_FILES = {
    "train": "/path",
    "validation": "/path",
    "test": "/path",
}

DATASET_SEGMENT_PREFIX = Path(INPUT_FILES["train"]).stem.rsplit("_train", 1)[0]
OUTPUT_DIR = "/content/summary_labels"

# ==================== OPENAI CONFIGURATION ====================

MODEL = "gpt-6-sol"

REASONING_EFFORT = "high"
TEMPERATURE = None
MAX_OUTPUT_TOKENS = 16000

PROMPT_VERSION = "" # V1, 2, ...

# ==================== GENERATION SETTINGS ====================

MAX_ATTEMPTS = 3
RETRY_DELAY_SECONDS = 5

RESUME = True
AUTO_DOWNLOAD_ZIP = True

# ==================== EVENT SCHEMA CONFIGURATION ====================

# Fields that each event must contain.
REQUIRED_EVENT_COLUMNS = {"Content"}

# Alternative names for event identifiers.
EVENT_ID_COLUMNS = (
    "LineId",
    "line_id",
    "lineId",
    "log_id",
)

# Alternative names for event ground-truth labels.
EVENT_LABEL_COLUMNS = (
    "Label",
    "label",
)

# Fields that should not be repeated as event metadata.
EXCLUDED_EVENT_COLUMNS = {
    "LineId", "line_id", "lineId", "log_id",
    "Label", "label",
    "Content",
    "EventTemplate", "event_template", "eventtemplate",
}

# Preserve available metadata across different systems.
# Examples: Timestamp, Node, Component, Level, PID, etc.
INCLUDE_EVENT_METADATA = True

# Include metadata names, e.g. Level=ERROR instead of ERROR.
INCLUDE_METADATA_KEYS = True

# Events without a ground-truth label receive this value.
UNKNOWN_LABEL_VALUE = "unknown"

# Missing event IDs should be generated deterministically
# within each segment.
GENERATE_MISSING_EVENT_IDS = True

# ==================== VALIDATION CONFIGURATION ====================

VALIDATION_VERSION = "sampled-reference-v11-no-event-template"


Prompt

In [ ]:
SYSTEMS = {
    "bgl": "IBM BlueGene/L supercomputer",
    "apache": "Apache HTTP server",
    "linux": "Linux system logs",
    "openstack": "OpenStack cloud",
    "zookeeper": "Apache ZooKeeper service",
}
SYSTEM_HINTS = {
    "bgl": "",
    "apache": "For unknown events, distinguish genuine web-server or application failures from expected request handling and routine operation. Treat an event as a fault only when the logs provide clear evidence of an unsuccessful internal operation, malfunction, or abnormal service behavior. Client-side errors, rejected requests, and diagnostic messages are not faults unless they reveal an underlying system problem. Consider the event context and outcome rather than severity or error keywords alone.",
    "linux": "'linux': 'Linux operating system logs covering kernel, hardware, system services, and user activities. For unknown events, distinguish genuine system malfunctions from routine operations, expected access rejections, and harmless diagnostics. Treat an event as a fault when logs provide clear evidence of an unsuccessful internal operation, system or service malfunction, or abnormal system behavior, including memory exhaustion, forced process termination, and resource allocation failures. Authentication failures, client disconnections, connection timeouts, and access denials are not system faults by themselves unless they reveal a malfunction or disrupted service. Kernel initialization, device configuration, audit records, and normal service lifecycle activities are generally not faults unless they indicate an actual failure. Evaluate the actual outcome and surrounding context rather than error keywords, warnings, or severity alone.",
    'zookeeper': 'Apache ZooKeeper distributed coordination service managing cluster membership, leader election, client sessions, and replicated data. For unknown events, distinguish genuine coordination or service failures from expected distributed-system behavior, transient connection issues, and routine operations. Treat an event as a fault when logs provide clear evidence of an unsuccessful internal operation, quorum or synchronization failure, service unavailability, data consistency problems, or abnormal server termination. Connection interruptions, election retries, session expirations, socket exceptions, and worker thread shutdowns are not necessarily faults unless they indicate an actual malfunction, failed operation, or disrupted service. Normal leader elections, reconnections, session management, and snapshot operations are generally not faults unless they reveal an underlying failure. Evaluate the actual outcome and surrounding context rather than warnings, exception messages, or severity alone.',
    'proxifier': 'Proxifier network proxy client managing application connections, proxy routing, and network traffic through HTTP, HTTPS, and SOCKS proxies. For unknown events, distinguish genuine proxy or network communication failures from routine connection handling, expected restrictions, and application-initiated actions. Treat an event as a fault when logs provide clear evidence of an unsuccessful connection or proxy operation caused by a communication problem, proxy unavailability, name resolution failure, or abnormal connection termination. Connection closures, short-lived sessions, zero-byte transfers, canceled requests, and access rejections are not necessarily faults unless they indicate an actual malfunction or disrupted operation. Normal connection establishment, traffic statistics, and rule-based routing or blocking are generally not faults unless they reveal an underlying failure. Evaluate the actual outcome and surrounding context rather than error keywords, status codes, or message appearance alone.',
    }
LABEL_RULES = """Each event has [label=...]. '-' means confirmed normal regardless of alarming words;
'unknown' requires independent judgment if it is normal or faulty; every other value means confirmed fault.
For unknown events, select a fault only when the message clearly shows a real failure
(e.g., crash, uncorrected hardware/I/O fault, unsuccessful operation). Corrected/recovered
notices, diagnostics, counters, periodic messages, and uncertain cases are not fault.
Known labels always override interpretation. Never expose [label=...] or call events
'label', 'normal', 'anomalous', or 'unlabeled' in your output; use fault/background.
'unknown' is allowed only as a schema value such as recovery_state or fault_category."""
CONTEXT_RULES = """Choose up to 3 informative background events: lifecycle transitions,
nearby same-component context, or rare changes. Exclude repeated routine activity,
status/counters, diagnostics, and already-corrected notices. Keep input order."""
PROMPTS_EXPLANATION = {
 "extractive": """You summarize consecutive logs from {SYSTEM}. {SYSTEM_HINTS}
{LABEL_RULES}
Provide a nonempty, brief explanation of the main selections. Event ID citations are optional; if no faults, begin explanation with "Everything looks normal."
critical_events: all distinct fault messages, one original representative each, in input order.
context_events: {CONTEXT_RULES}
Copy selected lines EXACTLY except remove [label=...]; retain [id=...].
explanation and lists must agree. Return JSON only:
{{"explanation":"...","critical_events":[],"context_events":[]}}""",
 "abstractive": """You summarize consecutive logs from {SYSTEM} for nonexperts. {SYSTEM_HINTS}
{LABEL_RULES}
Write a brief, nonempty explanation of the main selections; citations are optional. If no faults, begin explanation with "Everything looks normal." Qualify inferred faults.
Write ONE short paragraph: 'The fault:' followed by faults, affected components and
observed consequences; count repeats if useful. Then 'For background,' with only
qualifying context: {CONTEXT_RULES}
If no faults, start summary with "Everything looks normal." and optionally add relevant background.
Do not invent outcomes or copy full lines. Return JSON only:
{{"explanation":"...","summary":"..."}}""",
 "hybrid": """You summarize consecutive logs from {SYSTEM}. {SYSTEM_HINTS}
{LABEL_RULES}
Provide a nonempty, brief explanation of the main selections; event ID citations are optional. If no faults, begin explanation with "Everything looks normal." Qualify inferred faults.
critical_events: every distinct fault message (one exact original line per message).
context_events: {CONTEXT_RULES}
Copy selected lines exactly, removing only [label=...], preserving [id=...] and order.
When critical_events is nonempty, analysis must contain:
- fault_category: hardware, memory, storage_io, network, application, configuration, resource, security, unknown,
- affected_components: verbatim IDs,
- recovery_state: recovered, unrecovered, unknown,
- likely_cause: 'undetermined' if unsupported,
- evidence_ids: critical-event IDs only.
Add a 1-2 sentence interpretation without invented consequences. If no faults, analysis=null and interpretation="Everything looks normal."; context_events can be nonempty.
Return JSON only:
{{"explanation":"...","critical_events":[],"context_events":[],
"analysis":{{"fault_category":"...","affected_components":[],"recovery_state":"...",
"likely_cause":"...","evidence_ids":[]}},"interpretation":"..."}}""",
}
def build_prompts(templates, system_key):
    if system_key not in SYSTEMS:
        raise ValueError(f"Unsupported system: {system_key}")
    return {k: v.replace("{SYSTEM}", SYSTEMS[system_key])
             .replace("{SYSTEM_HINTS}", SYSTEM_HINTS[system_key])
             .replace("{LABEL_RULES}", LABEL_RULES)
             .replace("{CONTEXT_RULES}", CONTEXT_RULES)
             .replace('{{', '{').replace('}}', '}')
             for k, v in templates.items()}


## 2. Validation and API helpers
Deterministic checks enforce known-label constraints, exact source lines, evidence IDs and schema. Inferred unknown-event faults still require semantic review.

In [ ]:
LABEL_PATTERN = re.compile(
    r"\[label=([^\]\r\n]*)\]",
    flags=re.IGNORECASE,
)
ID_PATTERN = re.compile(r"\[id=([^\]\r\n]+)\]")
FAULT_CATEGORIES = {
    "hardware",
    "memory",
    "storage_io",
    "network",
    "application",
    "configuration",
    "resource",
    "security",
    "unknown",
}
RECOVERY_STATES = {
    "recovered",
    "unrecovered",
    "unknown",
}
GENERATED_COLUMNS = [
    "GT",
    "summary_type",
    "summary_model",
    "generation_config",
    "generation_status",
    "generation_error",
    "raw_response",
    "generation_time_seconds",
    "generation_attempts",
    "api_latency_seconds",
    "input_tokens",
    "output_tokens",
    "reasoning_tokens",
    "teacher_total_tokens",
    "teacher_visible_output_tokens",
    "unknown_event_decisions",
]
USE_JSON_MODE = USE_EXPLANATION or SUMMARY_TYPE in {"extractive", "hybrid"}
CURRENT_MESSAGE_BY_ID = {}
_last_raw_response = ""
_last_api_metrics = {}
_current_row = {}
# Holds the latest rejected response for a corrective retry.
_retry_state = {}
# ==================== 2. FILE HELPERS ====================
def save_csv_atomic(data, path):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    data.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig",
    )
    temporary_path.replace(path)
def append_attempt_record(record):
    """Keep raw responses and repaired outputs for review."""
    with attempt_log_file.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )
# ==================== 3. INPUT PARSING ====================
def whitespace_key(text):
    return re.sub(r"\s+", " ", text).strip()
def parse_chunk_events(chunk_text):
    """
    Read one event per line.
    Each event must contain exactly one [label=...] field.
    """
    events = []
    for line_number, line in enumerate(chunk_text.splitlines(), start=1):
        if not line.strip():
            continue
        matches = list(LABEL_PATTERN.finditer(line))
        if len(matches) != 1:
            raise ValueError(
                f"Input line {line_number} must contain exactly "
                "one [label=...] field."
            )
        match = matches[0]
        label = match.group(1).strip()
        if not label:
            raise ValueError(
                f"Input line {line_number} contains an empty label."
            )
        before = line[:match.start()]
        after = line[match.end():]
        # Preserve all text except the label and, optionally,
        # one adjacent separator space.
        canonical = before + after
        allowed_outputs = {canonical}
        if after.startswith(" "):
            allowed_outputs.add(before + after[1:])
        if before.endswith(" "):
            allowed_outputs.add(before[:-1] + after)
        ids = ID_PATTERN.findall(line)
        if len(ids) != 1 or ids[0] not in CURRENT_MESSAGE_BY_ID:
            raise ValueError("Each event must have one known [id=...] field.")
        events.append(
            {
                "original": line,
                "message": CURRENT_MESSAGE_BY_ID[ids[0]],
                "label": label,
                "canonical": canonical,
                "allowed_outputs": allowed_outputs,
            }
        )
    if not events:
        raise ValueError("The chunk contains no log events.")
    return events
# ==================== 4. SAFE REPAIRS ====================
def normalize_evidence_id(value):
    """Normalize representation without guessing or replacing an ID."""
    if type(value) is int:
        return str(value)
    if not isinstance(value, str) or not value.strip():
        raise ValueError(
            "Each evidence ID must be a nonempty string or integer; "
            f"received {value!r}."
        )
    value = value.strip()
    match = re.fullmatch(r"\[id=([^\]\r\n]+)\]", value)
    if match:
        value = match.group(1).strip()
    if not value:
        raise ValueError("An evidence ID is empty.")
    # Preserve leading zeros and nonnumeric identifiers.
    return value
def repair_json_summary(result, original_events):
    """
    Repair harmless output formatting.
    Return the repaired object and a list of repair descriptions.
    Never infer missing analysis or replace an unsupported event/ID.
    """
    repairs = []
    candidates = {}
    exact_forms = set()
    for original in original_events:
        canonical = original["canonical"]
        key = whitespace_key(canonical)
        candidates.setdefault(key, set()).add(canonical)
        exact_forms.update(original["allowed_outputs"])
    for field_name in ("critical_events", "context_events"):
        selected = result.get(field_name)
        if not isinstance(selected, list):
            continue
        for index, event in enumerate(selected):
            if not isinstance(event, str):
                continue
            without_label = LABEL_PATTERN.sub("", event)
            repaired = without_label
            # Keep an already valid exact representation.
            if without_label not in exact_forms:
                matches = candidates.get(
                    whitespace_key(without_label),
                    set(),
                )
                if len(matches) == 1:
                    repaired = next(iter(matches))
            if repaired != event:
                repairs.append(
                    f"{field_name}[{index}]: removed label "
                    "and/or restored source whitespace"
                )
            selected[index] = repaired
    analysis = result.get("analysis")
    if isinstance(analysis, dict):
        evidence_ids = analysis.get("evidence_ids")
        if isinstance(evidence_ids, list):
            normalized_ids = [
                normalize_evidence_id(value)
                for value in evidence_ids
            ]
            if normalized_ids != evidence_ids:
                repairs.append(
                    "analysis.evidence_ids: normalized ID formatting"
                )
            analysis["evidence_ids"] = normalized_ids
    return result, repairs
# ==================== 5. VALIDATION ====================
def require_string_list(value, field_name):
    if not isinstance(value, list):
        raise ValueError(f"{field_name} must be a list.")
    for item in value:
        if not isinstance(item, str) or not item.strip():
            raise ValueError(
                f"{field_name} must contain nonempty strings; "
                f"received {item!r}."
            )
def validate_event_list(
    selected,
    original_events,
    field_name,
    used_indices,
):
    """Check source membership, labels, order, and event reuse."""
    require_string_list(selected, field_name)
    if field_name == "context_events" and len(selected) > 3:
        raise ValueError("context_events may contain at most 3 events.")
    if len(selected) != len(set(selected)):
        raise ValueError(
            f"{field_name} contains duplicate event strings."
        )
    position = 0
    for event in selected:
        exact_matches = [
            index
            for index, original in enumerate(original_events)
            if event in original["allowed_outputs"]
        ]
        def eligible(index):
            label = original_events[index]["label"]
            if field_name == "context_events":
                return label in {"-", "unknown"}
            return label != "-"
        eligible_matches = [
            index for index in exact_matches if eligible(index)
        ]
        unused_matches = [
            index
            for index in eligible_matches
            if index not in used_indices
        ]
        ordered_matches = [
            index
            for index in unused_matches
            if index >= position
        ]
        if not ordered_matches:
            if not exact_matches:
                reason = (
                    "does not match an original event after safe "
                    "formatting repairs"
                )
            elif not eligible_matches:
                reason = "comes from a disallowed label group"
            elif not unused_matches:
                reason = "was already selected in another list"
            else:
                reason = "is out of the required input order"
            raise ValueError(
                f"{field_name}: {reason}. Event: {event!r}"
            )
        matched_index = ordered_matches[0]
        used_indices.add(matched_index)
        position = matched_index + 1
def validate_abstractive(text, original_events, fault_ids=None):
    """Validate basic abstractive summary structure without restricting wording."""
    if text in ('', '""'):
        raise ValueError("Abstractive summary must not be empty.")
    if "\n" in text or "\r" in text:
        raise ValueError(
            "The abstractive summary must be one paragraph."
        )
    if LABEL_PATTERN.search(text):
        raise ValueError(
            "The abstractive paragraph contains a label field."
        )
    return text
NORMAL_PHRASE = "Everything looks normal."


def validate_explanation(explanation, original_events=None, required_ids=None,
                         require_fault_citation=False, no_faults=False):
    """Require nonempty explanation and the normal phrase for no-fault segments.

    Do not reject based on event citations, length, or other explanation wording.
    """
    if not isinstance(explanation, str) or not explanation.strip():
        raise ValueError("explanation must be a nonempty string.")
    if no_faults and NORMAL_PHRASE not in explanation:
        raise ValueError(f"No-fault explanation must contain {NORMAL_PHRASE!r}.")

def validate_json_summary(result, original_events):
    if SUMMARY_TYPE == "abstractive":
        expected_keys = {"explanation", "summary"}
    else:
        expected_keys = {
            "critical_events",
            "context_events",
        }
        if SUMMARY_TYPE == "hybrid":
            expected_keys |= {"analysis", "interpretation"}
        if USE_EXPLANATION:
            expected_keys.add("explanation")
    if set(result) != expected_keys:
        raise ValueError(
            "Incorrect JSON keys. "
            f"Missing: {sorted(expected_keys - set(result))}; "
            f"extra: {sorted(set(result) - expected_keys)}."
        )
    if SUMMARY_TYPE == "abstractive":
        summary_text = result["summary"]
        if not isinstance(summary_text, str):
            raise ValueError("summary must be a string.")
        validate_abstractive(summary_text.strip(), original_events)
        validate_explanation(result["explanation"])
        return
    used_indices = set()
    validate_event_list(
        result["critical_events"],
        original_events,
        "critical_events",
        used_indices,
    )
    validate_event_list(
        result["context_events"],
        original_events,
        "context_events",
        used_indices,
    )
    critical = result["critical_events"]
    context = result["context_events"]
    if set(critical) & set(context):
        raise ValueError(
            "An event appears in both critical_events and context_events."
        )
    # Both prompts require every distinct anomalous MESSAGE exactly once.
    expected_messages = {e["message"] for e in original_events if e["label"] not in {"-", "unknown"}}
    selected_messages = [next(e["message"] for e in original_events
                              if line in e["allowed_outputs"]) for line in critical]
    if not expected_messages.issubset(set(selected_messages)):
        raise ValueError("critical_events must cover every distinct anomalous message exactly once.")
    if len(selected_messages) != len(set(selected_messages)):
        raise ValueError("critical_events repeats the same message under different event IDs.")
    if USE_EXPLANATION:
        selected_ids = {
            event_id
            for event in critical + context
            for event_id in ID_PATTERN.findall(event)
        }
        validate_explanation(result["explanation"], no_faults=not bool(critical))
    if SUMMARY_TYPE == "extractive":
        return
    # ---------- Hybrid without activities ----------
    analysis = result["analysis"]
    interpretation = result["interpretation"]
    if not isinstance(interpretation, str):
        raise ValueError("interpretation must be a string.")
    has_fault = any(
        event["label"] not in {"-", "unknown"}
        for event in original_events
    )
    if not critical:
        if has_fault:
            raise ValueError(
                "critical_events is empty despite non-'-' input events."
            )
        if analysis is not None:
            raise ValueError("Without critical events, analysis must be null.")
        if NORMAL_PHRASE not in interpretation:
            raise ValueError(f"No-fault hybrid interpretation must contain {NORMAL_PHRASE!r}.")
        return
    analysis_keys = {
        "fault_category",
        "affected_components",
        "recovery_state",
        "likely_cause",
        "evidence_ids",
    }
    if (
        not isinstance(analysis, dict)
        or set(analysis) != analysis_keys
    ):
        raise ValueError(
            "analysis must contain exactly: "
            + ", ".join(sorted(analysis_keys))
        )
    if (
        not isinstance(analysis["fault_category"], str)
        or analysis["fault_category"] not in FAULT_CATEGORIES
    ):
        raise ValueError("Invalid fault_category.")
    if (
        not isinstance(analysis["recovery_state"], str)
        or analysis["recovery_state"] not in RECOVERY_STATES
    ):
        raise ValueError("Invalid recovery_state.")
    require_string_list(
        analysis["affected_components"],
        "analysis.affected_components",
    )
    require_string_list(
        analysis["evidence_ids"],
        "analysis.evidence_ids",
    )
    if (
        not isinstance(analysis["likely_cause"], str)
        or not analysis["likely_cause"].strip()
    ):
        raise ValueError(
            "likely_cause must be a nonempty string. "
            "Use 'undetermined' if unsupported."
        )
    valid_ids = {
        event_id
        for event in critical
        for event_id in ID_PATTERN.findall(event)
    }
    evidence_ids = analysis["evidence_ids"]
    if not evidence_ids:
        raise ValueError(
            "analysis.evidence_ids cannot be empty "
            "when critical events exist."
        )
    invalid_ids = [
        event_id
        for event_id in evidence_ids
        if event_id not in valid_ids
    ]
    if invalid_ids:
        raise ValueError(
            f"Unsupported evidence IDs: {invalid_ids!r}. "
            f"Allowed IDs from retained critical_events: "
            f"{sorted(valid_ids)!r}."
        )
def validate_summary(text, chunk_text):
    """
    Return (validated_summary, repairs).
    These checks verify structure and source correspondence, not
    the full semantic correctness of the generated explanation.
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError(
            "The model returned no content. "
            'A valid empty abstractive result must explicitly be "".'
        )
    text = text.strip()
    original_events = parse_chunk_events(chunk_text)
    if SUMMARY_TYPE == "abstractive" and not USE_EXPLANATION:
        return validate_abstractive(text, original_events), []
    result = json.loads(text)
    if not isinstance(result, dict):
        raise ValueError("The response must be a JSON object.")
    result, repairs = repair_json_summary(
        result,
        original_events,
    )
    validate_json_summary(result, original_events)
    # The source text cannot contain derived template metadata. Reject any
    # template field unexpectedly emitted by the teacher, too.
    if re.search(r"(?i)(?<!\w)event[_\s-]*template\s*=", json.dumps(result, ensure_ascii=False)):
        raise ValueError("EventTemplate metadata leaked into generated summary")
    return json.dumps(result, ensure_ascii=False), repairs
# ==================== 6. API GENERATION ====================
def get_summary(chunk_text):
    global _last_raw_response, _last_api_metrics
    parse_chunk_events(chunk_text)
    chunk_hash = hashlib.sha256(
        chunk_text.encode("utf-8")
    ).hexdigest()
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": "Log chunk:\n" + chunk_text,
        },
    ]
    if _retry_state.get("chunk_hash") == chunk_hash:
        previous_response = _retry_state.get("response")
        if isinstance(previous_response, str) and previous_response:
            messages.append(
                {
                    "role": "assistant",
                    "content": previous_response,
                }
            )
        correction = (
            "Your previous response failed validation:\n"
            + _retry_state["error"]
            + "\n\nReturn the complete corrected response using the "
            "original required format. Fix the reported issue using "
            "only the supplied logs. Do not invent events, change "
            "identifiers, or add commentary outside the required output."
        )
        if SUMMARY_TYPE in {"extractive", "hybrid"}:
            correction += (
                " Copy selected events exactly, removing [label=...], "
                "and preserve their original order."
            )
        if USE_EXPLANATION:
            correction += (
                " Keep the explanation field first as a nonempty string. "
                "Event citations are optional. When no faults are selected, "
                "include the exact phrase 'Everything looks normal.' in the "
                "explanation and in the summary (abstractive) or interpretation "
                "(hybrid), as applicable."
            )
        if SUMMARY_TYPE == "hybrid":
            correction += (
                " Write evidence_ids as strings containing the values "
                "inside [id=...] from retained critical_events only. "
                "Do not cite context events or omitted events."
            )
        messages.append(
            {
                "role": "user",
                "content": correction,
            }
        )
    else:
        _retry_state.clear()
    request = {
        "model": MODEL,
        "messages": messages,
        "max_completion_tokens": MAX_OUTPUT_TOKENS,
    }
    if REASONING_EFFORT is not None:
        request["reasoning_effort"] = REASONING_EFFORT
    if TEMPERATURE is not None:
        request["temperature"] = TEMPERATURE
    if USE_JSON_MODE:
        request["response_format"] = {"type": "json_object"}
    started_api = time.perf_counter()
    response = client.chat.completions.create(**request)
    usage = getattr(response, "usage", None)
    details = getattr(usage, "completion_tokens_details", None)
    _last_api_metrics = {
        "api_latency_seconds": round(time.perf_counter() - started_api, 4),
        "input_tokens": getattr(usage, "prompt_tokens", "") if usage else "",
        "output_tokens": getattr(usage, "completion_tokens", "") if usage else "",
        "reasoning_tokens": getattr(details, "reasoning_tokens", "") if details else "",
    }
    if not response.choices:
        raise ValueError("The API returned no response choices.")
    choice = response.choices[0]
    message = choice.message
    raw_text = message.content
    _last_raw_response = raw_text or ""
    record = {
        **_current_row,
        "timestamp": time.time(),
        "generation_config": config_id,
        "summary_type": SUMMARY_TYPE,
        "chunk_hash": chunk_hash,
        "response_id": response.id,
        "finish_reason": choice.finish_reason,
        "raw_response": raw_text,
        "refusal": getattr(message, "refusal", None),
        **_last_api_metrics,
    }
    # Save before validation so rejected responses remain available.
    append_attempt_record(
        {**record, "stage": "received"}
    )
    try:
        if record["refusal"]:
            raise ValueError(
                f"Model refusal: {record['refusal']}"
            )
        if choice.finish_reason != "stop":
            raise ValueError(
                f"Generation ended with {choice.finish_reason!r}. "
                "If this is 'length', increase MAX_OUTPUT_TOKENS."
            )
        summary, repairs = validate_summary(
            raw_text,
            chunk_text,
        )
    except (ValueError, TypeError) as exc:
        _retry_state.clear()
        _retry_state.update(
            {
                "chunk_hash": chunk_hash,
                "response": raw_text,
                "error": str(exc),
            }
        )
        append_attempt_record(
            {
                **record,
                "stage": "validation_failed",
                "error": str(exc),
            }
        )
        raise
    _retry_state.clear()
    append_attempt_record(
        {
            **record,
            "stage": "completed",
            "repairs": repairs,
            "validated_summary": summary,
        }
    )
    if repairs:
        print(
            "Automatically repaired:\n- " + "\n- ".join(repairs),
            flush=True,
        )
    return summary


## 3. Load and check all three files (no API requests)

Event metadata is selected dynamically; EventId/template IDs are kept as metadata, not treated as unique line identifiers.


In [ ]:

# ============================================================
# CELL 10 — MULTI-DATASET INPUT VALIDATION AND PREPARATION
# Compatible with BGL, Apache, Linux, ZooKeeper
# ============================================================

# -------------------- 1. PROMPT CONFIGURATION --------------------

if not USE_EXPLANATION and "PROMPTS_NO_REASONING" not in globals():
    raise ValueError(
        "This notebook contains reasoning templates only. "
        "Set USE_EXPLANATION=True."
    )

PROMPTS = build_prompts(
    PROMPTS_EXPLANATION if USE_EXPLANATION else PROMPTS_NO_REASONING,
    SYSTEM_KEY
)

if SUMMARY_TYPE not in PROMPTS:
    raise ValueError(f"Choose one of {list(PROMPTS)}")

if MAX_ATTEMPTS < 1 or RETRY_DELAY_SECONDS < 0:
    raise ValueError("Invalid retry settings.")

if set(INPUT_FILES) != {"train", "validation", "test"}:
    raise ValueError(
        "INPUT_FILES must contain train, validation, and test."
    )

SYSTEM_PROMPT = PROMPTS[SUMMARY_TYPE]
USE_JSON_MODE = USE_EXPLANATION or SUMMARY_TYPE in {"extractive", "hybrid"}

print(
    f"System: {SYSTEM_KEY} | "
    f"Summary: {SUMMARY_TYPE} | "
    f"Reasoning: {USE_EXPLANATION}"
)


# -------------------- 2. CONFIGURATION FINGERPRINT --------------------

event_schema_config = {
    "required_event_columns": sorted(REQUIRED_EVENT_COLUMNS),
    "event_id_columns": list(EVENT_ID_COLUMNS),
    "event_label_columns": list(EVENT_LABEL_COLUMNS),
    "excluded_event_columns": sorted(EXCLUDED_EVENT_COLUMNS),
    "include_event_metadata": INCLUDE_EVENT_METADATA,
    "include_metadata_keys": INCLUDE_METADATA_KEYS,
    "unknown_label_value": UNKNOWN_LABEL_VALUE,
    "generate_missing_event_ids": GENERATE_MISSING_EVENT_IDS,
    "event_format_version": "dynamic-metadata-v4-eventtemplate-excluded",
}

config_id = hashlib.sha256(
    json.dumps(
        {
            "system_key": SYSTEM_KEY,
            "summary_type": SUMMARY_TYPE,
            "prompt": SYSTEM_PROMPT,
            "model": MODEL,
            "temperature": TEMPERATURE,
            "reasoning_effort": REASONING_EFFORT,
            "prompt_version": PROMPT_VERSION,
            "max_output_tokens": MAX_OUTPUT_TOKENS,
            "use_json_mode": USE_JSON_MODE,
            "validation_version": VALIDATION_VERSION,
            "event_schema": event_schema_config,
            "approach": APPROACH,
            "use_explanation": USE_EXPLANATION,
            "input_files": {
                k: str(v) for k, v in INPUT_FILES.items()
            },
        },
        sort_keys=True,
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()


# -------------------- 3. HELPER FUNCTIONS --------------------

def is_missing(value):
    """Identify missing metadata or label values."""
    if value is None:
        return True
    if isinstance(value, (dict, list)):
        return False
    try:
        result = pd.isna(value)
        if isinstance(result, (bool, type(pd.NA))):
            if result is pd.NA:
                return True
            if result:
                return True
    except (TypeError, ValueError):
        pass
    return isinstance(value, str) and not value.strip()


def safe_text(value):
    """Convert scalar values to text without creating nan/None."""
    if is_missing(value):
        return ""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return str(value)


def find_column(event, candidates):
    """Return the first available nonempty field and its value."""
    for key in candidates:
        if key in event and not is_missing(event[key]):
            return key, safe_text(event[key])
    return None, None


def get_event_label(event):
    """Read known ground truth or mark the event as unknown."""
    _, value = find_column(event, EVENT_LABEL_COLUMNS)

    if value is None:
        return UNKNOWN_LABEL_VALUE

    value = value.strip()
    return value if value else UNKNOWN_LABEL_VALUE


def validate_tag_value(value, field_name):
    """Prevent malformed or ambiguous structured tags."""
    if not value:
        raise ValueError(f"Empty {field_name}")

    if any(c in value for c in "[]\r\n"):
        raise ValueError(
            f"Invalid characters in {field_name}: {value!r}"
        )


def get_event_id(event, segment_id, position, split):
    """
    Prefer actual source LineId.

    Do not use EventId as a unique identifier: in many log
    datasets, EventId is a log-template identifier repeated
    across multiple source events.

    When LineId is missing, generate a deterministic ID
    namespaced by split, segment and position.
    """
    source_id_columns = EVENT_ID_COLUMNS

    _, value = find_column(event, source_id_columns)

    if value is not None:
        validate_tag_value(value, "event ID")
        return value, True

    if not GENERATE_MISSING_EVENT_IDS:
        raise ValueError(
            f"Missing source event ID in segment {segment_id}"
        )

    # Deterministic local ID for the generated log line.
    segment_digest = hashlib.sha256(
        str(segment_id).encode("utf-8")
    ).hexdigest()[:12]

    generated_id = (
        f"{split}_{segment_digest}_{position + 1}"
    )

    return generated_id, False


def is_excluded_event_field(key):
    """Exclude derived templates regardless of key case or underscores."""
    if key in EXCLUDED_EVENT_COLUMNS:
        return True
    normalized = re.sub(r"[^a-z0-9]", "", str(key).lower())
    return normalized == "eventtemplate"


def assert_no_template_metadata(text, location):
    """Catch template metadata escaping into model inputs or outputs."""
    if re.search(r"(?i)(?<!\w)event[_\s-]*template\s*=", text):
        raise ValueError(f"EventTemplate metadata detected in {location}")


def build_metadata(event):
    """
    Dynamically include metadata fields available in each
    system. Preserve their original dictionary order.
    """
    if not INCLUDE_EVENT_METADATA:
        return ""

    parts = []

    for key, value in event.items():

        if is_excluded_event_field(key):
            continue

        if is_missing(value):
            continue

        text_value = safe_text(value)

        # Avoid multiline metadata and reserved event tags.
        if "\n" in text_value or "\r" in text_value:
            raise ValueError(
                f"Multiline metadata field: {key}"
            )

        if LABEL_PATTERN.search(text_value) or ID_PATTERN.search(text_value):
            raise ValueError(
                f"Reserved event tag inside metadata: {key}"
            )

        if INCLUDE_METADATA_KEYS:
            parts.append(f"{key}={text_value}")
        else:
            parts.append(text_value)

    return " ".join(parts)


def make_event_line(event, eid, label):
    """
    Construct one source event for teacher input.

    Keep the original Content text unchanged.
    Add available metadata and normalized ID/label tags.
    """
    content = safe_text(event["Content"])

    if not content.strip():
        raise ValueError("Empty event Content")

    metadata = build_metadata(event)

    parts = [
        f"[id={eid}]",
        f"[label={label}]",
    ]

    if metadata:
        parts.append(metadata)

    parts.append(content)

    line = " ".join(parts)

    if "\n" in line or "\r" in line:
        raise ValueError(
            "Expected one original log event per line"
        )

    if len(LABEL_PATTERN.findall(line)) != 1:
        raise ValueError("Ambiguous label tags")

    if len(ID_PATTERN.findall(line)) != 1:
        raise ValueError("Ambiguous event ID tags")

    assert_no_template_metadata(line, "prepared teacher event")
    return line


# -------------------- 4. LOAD AND VALIDATE DATA --------------------

def prepare_source(path, split):

    if not path.is_file():
        raise FileNotFoundError(
            f"Upload the {split} CSV or correct its path: {path}"
        )

    frame = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig"
    )

    # Segment-level required fields.
    required = {"segment_id", "events_json"}

    if frame.empty or not required.issubset(frame.columns):
        raise ValueError(
            f"{path.name}: expected a nonempty CSV "
            f"containing {required}"
        )

    # Normalize segment identifiers.
    frame["segment_id"] = frame["segment_id"].str.strip()

    if (
        frame["segment_id"].eq("").any()
        or frame["segment_id"].duplicated().any()
    ):
        raise ValueError(
            f"{path.name}: empty or duplicate segment IDs"
        )

    if "split" in frame.columns:
        if not frame["split"].eq(split).all():
            raise ValueError(
                f"{path.name}: split field does not match {split}"
            )

    all_source_ids = set()

    # Store IDs for each chunk, including generated IDs.
    chunk_event_ids = []

    # Audit information.
    known_fault_counts = []
    unknown_event_counts = []
    event_id_sources = []
    prepared_id_message_maps = []

    for index, row in frame.iterrows():

        segment_id = row["segment_id"]

        try:
            records = json.loads(row["events_json"])
        except (json.JSONDecodeError, TypeError) as exc:
            raise ValueError(
                f"{path.name} row {index}: invalid events_json"
            ) from exc

        if not isinstance(records, list) or not records:
            raise ValueError(
                f"{path.name} row {index}: "
                "events_json must be a nonempty list"
            )

        lines = []
        messages = []
        labels = []
        ids = []
        source_flags = []

        for position, event in enumerate(records):

            if not isinstance(event, dict):
                raise ValueError(
                    f"{path.name} row {index}: "
                    "event must be a JSON object"
                )

            # Content is required; other metadata is optional.
            if not REQUIRED_EVENT_COLUMNS.issubset(event):
                missing = REQUIRED_EVENT_COLUMNS - set(event)
                raise ValueError(
                    f"{path.name} row {index}: "
                    f"missing event fields {missing}"
                )

            if not isinstance(event["Content"], str):
                raise ValueError(
                    f"{path.name} row {index}: "
                    "Content must be a string"
                )

            eid, has_source_id = get_event_id(
                event, segment_id, position, split
            )

            label = get_event_label(event)

            validate_tag_value(label, "event label")

            line = make_event_line(event, eid, label)

            lines.append(line)
            messages.append(event["Content"])
            labels.append(label)
            ids.append(eid)
            source_flags.append(has_source_id)

        # Every event must have a unique ID within its chunk.
        if len(set(ids)) != len(ids):
            raise ValueError(
                f"{path.name} row {index}: repeated event IDs"
            )

        # Preserve original source-ID overlap detection.
        actual_source_ids = {
            eid for eid, is_source in zip(ids, source_flags)
            if is_source
        }

        if all_source_ids.intersection(actual_source_ids):
            raise ValueError(
                f"{path.name}: repeated source event IDs"
            )

        all_source_ids.update(actual_source_ids)

        anomaly_count = sum(
            label not in {"-", UNKNOWN_LABEL_VALUE}
            for label in labels
        )

        unknown_count = sum(
            label == UNKNOWN_LABEL_VALUE
            for label in labels
        )

        # -------------------- SEGMENT LABEL CHECK --------------------

        # Ground truth is optional for partially labeled datasets.
        #
        # With unknown events, a chunk may contain inferred faults
        # not represented in its supplied status_gt_label.
        # Therefore, only fully labeled chunks are checked
        # for exact normal/anomalous status consistency.

        if unknown_count == 0:

            expected_status = (
                "anomalous" if anomaly_count else "normal"
            )

            if "status_gt_label" in frame.columns:

                supplied_status = row["status_gt_label"].strip().lower()

                if (
                    supplied_status
                    and supplied_status != expected_status
                ):
                    raise ValueError(
                        f"{path.name} segment {segment_id}: "
                        f"expected status {expected_status}, "
                        f"found {supplied_status}"
                    )

        # -------------------- COUNT CHECKS --------------------

        # Event count remains verifiable even with unknown labels.
        if "event_count" in frame.columns and row["event_count"].strip():
            if int(row["event_count"]) != len(records):
                raise ValueError(
                    f"{path.name} segment {segment_id}: "
                    "inconsistent event_count"
                )

        # The number of known anomalous labels is reliable,
        # but some datasets may omit unknown events from this
        # count. Only enforce equality for fully labeled chunks.
        if (
            "anomalous_event_count" in frame.columns
            and unknown_count == 0
            and row["anomalous_event_count"].strip()
        ):
            if int(row["anomalous_event_count"]) != anomaly_count:
                raise ValueError(
                    f"{path.name} segment {segment_id}: "
                    "inconsistent anomalous_event_count"
                )

        if (
            "event_labels_json" in frame.columns
            and row["event_labels_json"].strip()
        ):
            supplied_labels = json.loads(
                row["event_labels_json"]
            )

            supplied_labels = [
                safe_text(x).strip() or UNKNOWN_LABEL_VALUE
                for x in supplied_labels
            ]

            if supplied_labels != labels:
                raise ValueError(
                    f"{path.name} segment {segment_id}: "
                    "event_labels_json disagrees with events_json"
                )

        # Prevent derived template metadata from reaching the API or student.
        for prepared_line in lines:
            assert_no_template_metadata(prepared_line, f"{path.name}:{segment_id}")

        # -------------------- PREPARED INPUTS --------------------

        # Teacher input: known labels + unknown markers.
        frame.at[index, "Content"] = "\n".join(lines)

        # Raw message text, without metadata.
        frame.at[index, "messages_only"] = "\n".join(messages)

        # Student input: same events, with labels removed.
        frame.at[index, "Content_without_labels"] = "\n".join(
            LABEL_PATTERN.sub("", line, count=1)
            .replace("]  ", "] ", 1)
            for line in lines
        )

        chunk_event_ids.append(ids)
        prepared_id_message_maps.append(dict(zip(ids, messages)))
        known_fault_counts.append(anomaly_count)
        unknown_event_counts.append(unknown_count)

        event_id_sources.append(
            "source" if all(source_flags)
            else "generated" if not any(source_flags)
            else "mixed"
        )

    # Add useful auditing metadata.
    frame["known_fault_count"] = known_fault_counts
    frame["unknown_event_count"] = unknown_event_counts
    frame["event_id_source"] = event_id_sources
    # Internal mapping needed by validation, including generated IDs.
    frame["_id_message_map_json"] = [
        json.dumps(mapping, ensure_ascii=False)
        for mapping in prepared_id_message_maps
    ]

    return frame, all_source_ids, chunk_event_ids


# -------------------- 5. PREPARE ALL PARTITIONS --------------------

prepared_sources = {}
source_ids = {}
chunk_ids = {}

for split, filename in INPUT_FILES.items():

    (
        prepared_sources[split],
        source_ids[split],
        chunk_ids[split],
    ) = prepare_source(Path(filename), split)


# -------------------- 6. CHECK SOURCE EVENT OVERLAP --------------------

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    shared_ids = source_ids[a] & source_ids[b]

    if shared_ids:
        raise ValueError(
            f"Shared source events between {a} and {b}: "
            f"{len(shared_ids)} IDs overlap. "
            "Check the input partitions."
        )


# -------------------- 7. REPORT --------------------

print("\nPreflight passed.")
print("Dataset:", SYSTEM_KEY)
print("Configuration ID:", config_id[:16])
print("Full-event Content rebuilt from events_json.\n")

for split, frame in prepared_sources.items():

    print(f"{split}: {len(frame)} segments")

    if "status_gt_label" in frame.columns:
        print(
            "  GT status:",
            frame["status_gt_label"].value_counts().to_dict()
        )

    print(
        "  Known fault events:",
        int(frame["known_fault_count"].sum())
    )

    print(
        "  Unknown events:",
        int(frame["unknown_event_count"].sum())
    )

    print(
        "  Event IDs:",
        frame["event_id_source"].value_counts().to_dict()
    )

print(
    "\nExample teacher input:\n",
    prepared_sources["train"].iloc[0]["Content"]
)


## 4. Resumable per-split processing


In [ ]:
def process_split(split, source):
    global output_file, checkpoint_file, attempt_log_file
    global CURRENT_MESSAGE_BY_ID, _last_raw_response, _last_api_metrics, _current_row
    output_file = run_dir / f"{Path(INPUT_FILES[split]).stem}_{SUMMARY_TYPE}.csv"
    checkpoint_file = run_dir / f"{Path(INPUT_FILES[split]).stem}_{SUMMARY_TYPE}_checkpoint.csv"
    attempt_log_file = output_file.with_suffix(".attempts.jsonl")
    base = source.drop(
        columns=GENERATED_COLUMNS,
        errors="ignore",
    ).copy()
    df = base.copy()
    df["GT"] = ""
    df["summary_type"] = SUMMARY_TYPE
    df["summary_model"] = MODEL
    df["generation_config"] = config_id
    df["generation_status"] = "pending"
    df["generation_error"] = ""
    df["raw_response"] = ""
    df["generation_time_seconds"] = ""
    df["generation_attempts"] = ""
    for col in ["api_latency_seconds", "input_tokens", "output_tokens", "reasoning_tokens", "teacher_total_tokens", "teacher_visible_output_tokens", "unknown_event_decisions"]:
        df[col] = ""
    # ==================== 8. RESTORE PROGRESS ====================
    resume_path = (
        checkpoint_file
        if checkpoint_file.exists()
        else output_file
        if output_file.exists()
        else None
    )
    if RESUME and resume_path is not None:
        previous = pd.read_csv(
            resume_path,
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig",
        )
        required_columns = list(base.columns) + GENERATED_COLUMNS
        if not set(required_columns).issubset(previous.columns):
            raise ValueError(
                "Saved results are missing required columns. "
                "Use a different OUTPUT_DIR or set RESUME=False."
            )
        if not previous[list(base.columns)].equals(base):
            raise ValueError(
                "Saved results do not match the current input. "
                "Use a different OUTPUT_DIR or set RESUME=False."
            )
        if (
            not previous["generation_config"].eq(config_id).all()
            or not previous["summary_type"].eq(SUMMARY_TYPE).all()
            or not previous["summary_model"].eq(MODEL).all()
        ):
            raise ValueError(
                "Saved results use different prompts or settings. "
                "Use a different OUTPUT_DIR to preserve them, or "
                "set RESUME=False to regenerate and overwrite."
            )
        df = previous[required_columns].copy()
        print(f"Restored progress from: {resume_path}", flush=True)
        invalid_saved_count = 0
        repaired_saved_count = 0
        for index in df.index[df["generation_status"].eq("completed")]:
            CURRENT_MESSAGE_BY_ID = json.loads(df.at[index, "_id_message_map_json"])
            saved_summary = df.at[index, "GT"]
            try:
                validated, repairs = validate_summary(
                    saved_summary,
                    df.at[index, "Content"],
                )
                df.at[index, "GT"] = validated
                df.at[index, "generation_error"] = ""
                if repairs:
                    repaired_saved_count += 1
                    append_attempt_record(
                        {
                            "timestamp": time.time(),
                            "stage": "checkpoint_repaired",
                            "generation_config": config_id,
                            "row_index": int(index),
                            "raw_response": saved_summary,
                            "validated_summary": validated,
                            "repairs": repairs,
                        }
                    )
            except (ValueError, TypeError) as exc:
                append_attempt_record(
                    {
                        "timestamp": time.time(),
                        "stage": "checkpoint_validation_failed",
                        "generation_config": config_id,
                        "row_index": int(index),
                        "raw_response": saved_summary,
                        "error": str(exc),
                    }
                )
                df.at[index, "GT"] = ""
                df.at[index, "generation_status"] = "pending"
                df.at[index, "generation_error"] = str(exc)
                invalid_saved_count += 1
        if repaired_saved_count:
            print(
                f"Saved summaries repaired locally: {repaired_saved_count:,}",
                flush=True,
            )
        if invalid_saved_count:
            print(
                f"Saved summaries requiring regeneration: "
                f"{invalid_saved_count:,}",
                flush=True,
            )
    # ==================== 9. PROCESS UNFINISHED CHUNKS ====================
    completed = df["generation_status"].eq("completed")
    pending_indices = df.index[~completed].tolist()
    print(f"\nSummary type: {SUMMARY_TYPE}", flush=True)
    print(f"Total chunks: {len(df):,}", flush=True)
    print(f"Already completed: {int(completed.sum()):,}", flush=True)
    print(f"Remaining: {len(pending_indices):,}", flush=True)
    print(f"Attempt log: {attempt_log_file}", flush=True)
    try:
        for index in pending_indices:
            print(
                f"\nProcessing chunk {index + 1}/{len(df)}...",
                flush=True,
            )
            _retry_state.clear()
            _last_raw_response = ""
            _last_api_metrics = {}
            metrics_totals = {"api_latency_seconds": 0., "input_tokens": 0, "output_tokens": 0, "reasoning_tokens": 0}
            CURRENT_MESSAGE_BY_ID = json.loads(df.at[index, "_id_message_map_json"])
            _current_row = {"approach": APPROACH, "split": split,
                            "row_index": int(index), "segment_id": df.at[index, "segment_id"]}
            df.at[index, "GT"] = ""
            df.at[index, "generation_status"] = "pending"
            df.at[index, "generation_error"] = ""
            chunk_text = df.at[index, "Content"]
            # Invalid input should not trigger paid API retries.
            try:
                parse_chunk_events(chunk_text)
            except ValueError as exc:
                df.at[index, "generation_status"] = "failed"
                df.at[index, "generation_error"] = f"Invalid input: {exc}"
                print(f"Invalid input: {exc}", flush=True)
                save_csv_atomic(df, checkpoint_file)
                continue
            chunk_started = time.perf_counter()
            for attempt in range(1, MAX_ATTEMPTS + 1):
                df.at[index, "generation_attempts"] = attempt
                try:
                    summary = get_summary(chunk_text)
                    df.at[index, "raw_response"] = _last_raw_response
                    df.at[index, "GT"] = summary
                    df.at[index, "generation_status"] = "completed"
                    df.at[index, "generation_error"] = ""
                    print(f"Summary:\n{summary}", flush=True)
                    break
                except Exception as exc:
                    df.at[index, "raw_response"] = _last_raw_response
                    error_message = f"{type(exc).__name__}: {exc}"
                    df.at[index, "generation_error"] = error_message
                    fatal_request_error = (
                        getattr(exc, "status_code", None)
                        in {400, 401, 403, 404}
                    )
                    append_attempt_record({**_current_row, "timestamp": time.time(),
                                           "generation_config": config_id, "stage": "attempt_error",
                                           "attempt": attempt, "error": error_message})
                    if fatal_request_error or attempt == MAX_ATTEMPTS:
                        df.at[index, "generation_status"] = "failed"
                    print(
                        f"Attempt {attempt}/{MAX_ATTEMPTS} failed: {exc}",
                        flush=True,
                    )
                    if fatal_request_error:
                        raise
                    if attempt < MAX_ATTEMPTS:
                        time.sleep(RETRY_DELAY_SECONDS * attempt)
                finally:
                    for metric in metrics_totals:
                        metrics_totals[metric] += (_last_api_metrics.get(metric) or 0)
                    _last_api_metrics = {}
            for metric, value in metrics_totals.items():
                df.at[index, metric] = round(value, 4) if metric == "api_latency_seconds" else value
            # API output_tokens includes internal reasoning tokens when the model reports them.
            # Counts include all API attempts for this segment, including retries.
            df.at[index, "teacher_total_tokens"] = metrics_totals["input_tokens"] + metrics_totals["output_tokens"]
            df.at[index, "teacher_visible_output_tokens"] = max(
                0, metrics_totals["output_tokens"] - metrics_totals["reasoning_tokens"]
            )
            if df.at[index, "generation_status"] == "completed":
                decoded = json.loads(df.at[index, "GT"])
                events = parse_chunk_events(chunk_text)
                if SUMMARY_TYPE == "abstractive":
                    # Abstractive output has no per-event evidence IDs; do not
                    # fabricate event-level decisions from paragraph text.
                    df.at[index, "unknown_event_decisions"] = json.dumps([], ensure_ascii=False)
                else:
                    fault_ids = [v for line in decoded["critical_events"] for v in ID_PATTERN.findall(line)]
                    context_ids = [v for line in decoded["context_events"] for v in ID_PATTERN.findall(line)]
                    df.at[index, "unknown_event_decisions"] = json.dumps([
                        {"id": ID_PATTERN.findall(e["original"])[0],
                         "decision": ("fault" if ID_PATTERN.findall(e["original"])[0] in fault_ids else
                                      "background" if ID_PATTERN.findall(e["original"])[0] in context_ids else "not_selected"),
                         "source": "teacher_inferred"}
                        for e in events if e["label"] == "unknown"
                    ], ensure_ascii=False)
            df.at[index, "generation_time_seconds"] = round(time.perf_counter() - chunk_started, 4)
            print(f"Generation time (including retries): {df.at[index, 'generation_time_seconds']} s", flush=True)
            save_csv_atomic(df, checkpoint_file)
    finally:
        # Save progress even when the notebook cell is interrupted.
        save_csv_atomic(df, checkpoint_file)
        save_csv_atomic(df, output_file)

    counts = df.generation_status.value_counts().to_dict()
    print(f"{split}: {counts}; saved {output_file}")
    return {"approach": APPROACH, "split": split, "summary_type": SUMMARY_TYPE,
            "total": len(df), "completed": counts.get("completed", 0),
            "failed": counts.get("failed", 0), "pending": counts.get("pending", 0),
            "output_file": str(output_file)}


## 5. Generate reference summaries for all three splits


In [ ]:
# This is the only cell that sends paid API requests.
# Every unfinished row in all three files is processed. No resplitting/resampling.
openai.api_key = "---"
api_key = os.environ.get("OPENAI_API_KEY") or openai.api_key
if not api_key:
    api_key = getpass("Enter your OpenAI API key: ").strip()
if not api_key:
    raise ValueError("An OpenAI API key is required.")
client = OpenAI(api_key=api_key, max_retries=0, timeout=90.0)
run_dir = Path(OUTPUT_DIR) / APPROACH / f"{DATASET_SEGMENT_PREFIX}_{RUN_TAG}"
run_dir.mkdir(parents=True, exist_ok=True)
run_report = []
try:
    for split in ["train", "validation", "test"]:
        run_report.append(process_split(split, prepared_sources[split]))
finally:
    if run_report:
        save_csv_atomic(pd.DataFrame(run_report), run_dir / "generation_report.csv")
print(pd.DataFrame(run_report).to_string(index=False))


## 6. Download outputs and audit logs


In [ ]:
# Include CSV outputs, checkpoints, and raw-response audit logs.
archive = run_dir.parent / f"{DATASET_SEGMENT_PREFIX}_{APPROACH}_{RUN_TAG}_labels.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(run_dir.glob("*")):
        if path.is_file() and not path.name.endswith(".tmp"):
            bundle.write(path, arcname=path.name)
print(f"Results: {run_dir}")
print(f"ZIP: {archive}")
if AUTO_DOWNLOAD_ZIP:
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import display, FileLink
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))
